# PPMI Dataset Pre-processing

### Participant_Status

In [1]:
# Libraries
import pandas as pd

# Allow pandas to show all columns
pd.set_option('display.max_columns', None)

In [2]:
# Load the dataframe
df = pd.read_csv("../../ppmi_pd/Participant_Status_14Dec2025.csv")

# Display info and head to understand columns
print(df.info())
print(df.head())

# Also print all column names to help with categorization
print(df.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7905 entries, 0 to 7904
Data columns (total 30 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   PATNO               7905 non-null   int64  
 1   COHORT              7905 non-null   int64  
 2   COHORT_DEFINITION   7905 non-null   object 
 3   ENROLL_DATE         4509 non-null   object 
 4   ENROLL_STATUS       7905 non-null   object 
 5   STATUS_DATE         7904 non-null   object 
 6   SCREENEDAM          5816 non-null   float64
 7   ENROLL_AGE          4498 non-null   float64
 8   INEXPAGE            5820 non-null   object 
 9   AV133STDY           7058 non-null   float64
 10  TAUSTDY             7052 non-null   float64
 11  GAITSTDY            7052 non-null   float64
 12  PISTDY              7052 non-null   float64
 13  SV2ASTDY            7052 non-null   float64
 14  NXTAUSTDY           7052 non-null   float64
 15  DPPDSTDY            7052 non-null   float64
 16  DPPROS

In [3]:
# Load data dictionary and create column annotations
dd_df = pd.read_csv("../../ppmi_pd/Data_Dictionary_-__Annotated__23May2025.csv")

# Filter for PATIENT_STATUS module first (primary source)
patient_status_dd = dd_df[dd_df['MOD_NAME'] == 'PATIENT_STATUS'].copy()

# Create a dictionary mapping column names (ITM_NAME) to descriptions (DSCR)
# Handle cases where ITM_NAME might be empty or spaces
column_annotations = {}
for idx, row in patient_status_dd.iterrows():
    itm_name = str(row['ITM_NAME']).strip()
    dscr = str(row['DSCR']).strip()
    
    # Skip empty ITM_NAME (these are module header rows)
    if itm_name and itm_name != 'nan' and itm_name != '':
        column_annotations[itm_name] = dscr if dscr != 'nan' else ''

# For any columns in df that are not found in PATIENT_STATUS, search across all modules
for col in df.columns:
    if col not in column_annotations:
        # Search for this column name across all modules
        col_rows = dd_df[dd_df['ITM_NAME'] == col]
        if not col_rows.empty:
            # Take the first match (or could prefer PATIENT_STATUS if multiple exist)
            dscr = str(col_rows.iloc[0]['DSCR']).strip()
            column_annotations[col] = dscr if dscr != 'nan' else ''

# Print column annotations for verification
print("Column Annotations Found:")
print("-" * 80)
for col in df.columns:
    if col in column_annotations and column_annotations[col]:
        print(f"{col:25s}: {column_annotations[col]}")
    else:
        print(f"{col:25s}: [Description not found in data dictionary]")
print("-" * 80)

# Store column annotations as a dictionary attribute for later use
df.attrs['column_descriptions'] = column_annotations


Column Annotations Found:
--------------------------------------------------------------------------------
PATNO                    : Participant ID
COHORT                   : Enrollment Cohort
COHORT_DEFINITION        : Decoded Value for COHORT
ENROLL_DATE              : Enrollment Date
ENROLL_STATUS            : Enrollment Status
STATUS_DATE              : Date Enrollment Status Occurred
SCREENEDAM               : [Description not found in data dictionary]
ENROLL_AGE               : Age at Enrollment
INEXPAGE                 : PPMI Clinical Inclusion/Criteria
AV133STDY                : Participant in PPMI Clinical Early Imaging Sub-Study
TAUSTDY                  : Participant in PPMI Clinical Tau Imaging Sub-Study
GAITSTDY                 : Participant in PPMI Clinical Gait Sub-Study
PISTDY                   : Participant in PPMI Clinical Prodromal Imaging Sub-Study
SV2ASTDY                 : Participant in PPMI Clinical SV2A PET Sub-Study
NXTAUSTDY                : [Description not 

In [4]:
print(df['COHORT_DEFINITION'].value_counts())

COHORT_DEFINITION
Prodromal              5400
Parkinson's Disease    1984
Healthy Control         440
SWEDD                    81
Name: count, dtype: int64


In [5]:
print(df['ENROLL_STATUS'].value_counts())

ENROLL_STATUS
Enrolled             3717
Screen failed        2760
Withdrew              541
Excluded              271
Screened              120
Declined              114
Complete              102
Pending                94
Withdraw Deceased      73
Baseline Withdraw      44
Screen Scheduled       38
Baseline               26
LP Eligible             4
Lost to follow-up       1
Name: count, dtype: int64


In [6]:
# Function to create the master key dataframe
def create_master_key(file_path):
    # 1. Load the raw status file
    df = pd.read_csv(file_path)

    # 2. Define the exact columns you requested
    columns_to_keep = [
        'PATNO', 
        'COHORT', 
        'COHORT_DEFINITION', 
        'ENROLL_DATE', 
        'ENROLL_STATUS',
        'STATUS_DATE',
        'ENROLL_AGE',
        'AV133STDY', 
        'TAUSTDY', 
        'GAITSTDY', 
        'PISTDY', 
        'SV2ASTDY', 
        'NXTAUSTDY', 
        'DPPDSTDY', 
        'DPPROSTDY', 
        'FD4STDY', 
        'DATELIG', 
        'PPMI_ONLINE_ENROLL', 
        'ENRLPINK1', 
        'ENRLPRKN', 
        'ENRLSRDC', 
        'ENRLNORM', 
        'ENRLOTHGV', 
        'ENRLHPSM', 
        'ENRLRBD', 
        'ENRLLRRK2', 
        'ENRLSNCA', 
        'ENRLGBA'
    ]

    # 3. Create the subset DataFrame
    master_df = df[columns_to_keep].copy()

    # ---------------------------------------------------------
    # FILTER 1: ENROLL_STATUS
    # ---------------------------------------------------------
    # Keeping only valid participants with longitudinal potential
    valid_statuses = [
        'Complete', 
        'Enrolled', 
        'Withdraw Deceased', 
        'Withdrew'
    ]
    master_df = master_df[master_df['ENROLL_STATUS'].isin(valid_statuses)]

    # ---------------------------------------------------------
    # FILTER 2: COHORT_DEFINITION
    # ---------------------------------------------------------
    # Mapping "PD" to the CSV value "Parkinson's Disease"
    # This automatically excludes SWEDD and other non-relevant cohorts
    valid_cohorts = [
        'Healthy Control', 
        "Parkinson's Disease", 
        'Prodromal'
    ]
    master_df = master_df[master_df['COHORT_DEFINITION'].isin(valid_cohorts)]

    # ---------------------------------------------------------
    # FILTER 3: ENROLL_DATE
    # ---------------------------------------------------------
    # Remove rows where Enrollment Date is Missing (Null)
    master_df = master_df.dropna(subset=['ENROLL_DATE'])

    # ---------------------------------------------------------
    # CLEANUP
    # ---------------------------------------------------------
    # Convert ENROLL_DATE to actual datetime objects for math calculations
    master_df['ENROLL_DATE'] = pd.to_datetime(master_df['ENROLL_DATE'], format='%m/%Y')    
    # Reset index for a clean dataframe
    master_df.reset_index(drop=True, inplace=True)

    # ---------------------------------------------------------
    # FILTER 4: STATUS_DATE
    # ---------------------------------------------------------
    # Remove rows where Status Date is Missing (Null)
    master_df = master_df.dropna(subset=['STATUS_DATE'])

    # ---------------------------------------------------------
    # CLEANUP
    # ---------------------------------------------------------
    # Convert STATUS_DATE to actual datetime objects for math calculations
    master_df['STATUS_DATE'] = pd.to_datetime(master_df['STATUS_DATE'], format='%m/%Y')    
    # Reset index for a clean dataframe
    master_df.reset_index(drop=True, inplace=True)


    return master_df

# --- EXECUTION ---
file_path = '../../ppmi_pd/Participant_Status_14Dec2025.csv'
master_df = create_master_key(file_path)

# --- VERIFICATION ---
print(f"Final Master Key Shape: {master_df.shape}")
print("\nCohort Breakdown:")
print(master_df['COHORT_DEFINITION'].value_counts())
print("\nStatus Breakdown:")
print(master_df['ENROLL_STATUS'].value_counts())

Final Master Key Shape: (4369, 28)

Cohort Breakdown:
COHORT_DEFINITION
Prodromal              2546
Parkinson's Disease    1488
Healthy Control         335
Name: count, dtype: int64

Status Breakdown:
ENROLL_STATUS
Enrolled             3716
Withdrew              528
Withdraw Deceased      73
Complete               52
Name: count, dtype: int64


In [7]:
master_df.head(20)

,PATNO,COHORT,COHORT_DEFINITION,ENROLL_DATE,ENROLL_STATUS,STATUS_DATE,ENROLL_AGE,AV133STDY,TAUSTDY,GAITSTDY,PISTDY,SV2ASTDY,NXTAUSTDY,DPPDSTDY,DPPROSTDY,FD4STDY,DATELIG,PPMI_ONLINE_ENROLL,ENRLPINK1,ENRLPRKN,ENRLSRDC,ENRLNORM,ENRLOTHGV,ENRLHPSM,ENRLRBD,ENRLLRRK2,ENRLSNCA,ENRLGBA
0,3000,2,Healthy Control,2011-02-01,Withdrew,2024-10-01,69.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NO,0.0,0.0,0.0,NaN,NaN,0,0,0,0,0
1,3001,1,Parkinson's Disease,2011-03-01,Enrolled,2021-09-01,65.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NO,0.0,0.0,1.0,NaN,NaN,0,0,0,0,0
2,3002,1,Parkinson's Disease,2011-03-01,Withdrew,2024-10-01,67.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NO,0.0,0.0,1.0,NaN,NaN,0,0,0,0,0
3,3003,1,Parkinson's Disease,2011-04-01,Enrolled,2022-01-01,56.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,YES,0.0,0.0,1.0,NaN,NaN,0,0,0,0,0
4,3004,2,Healthy Control,2011-04-01,Enrolled,2022-01-01,59.4,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,YES,0.0,0.0,0.0,NaN,NaN,0,0,0,0,0
5,3006,1,Parkinson's Disease,2011-05-01,Withdrew,2013-10-01,57.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
6,3007,1,Parkinson's Disease,2011-05-01,Withdrew,2011-06-01,64.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NO,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
7,3008,2,Healthy Control,2011-06-01,Withdrew,2024-04-01,81.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NO,0.0,0.0,0.0,NaN,NaN,0,0,0,0,0
8,3009,2,Healthy Control,2011-06-01,Enrolled,2021-05-01,83.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,YES,0.0,0.0,0.0,NaN,NaN,0,0,0,0,0
9,3010,1,Parkinson's Disease,2011-06-01,Enrolled,2021-05-01,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,YES,0.0,0.0,1.0,NaN,NaN,0,0,0,0,0


## MOTOR__MDS-UPDRS

### Modified_Schwab

In [8]:
def create_schwab_df(file_path):
    # 1. Load the raw Schwab & England file
    schwab_df = pd.read_csv(file_path)

    # 2. Define the exact columns needed
    columns_to_keep = [
        'REC_ID',
        'PATNO',
        'EVENT_ID',
        'PAG_NAME',
        'INFODT',
        'MSEADLG',
        'LAST_UPDATE'
    ]
    
    # 3. Create the subset DataFrame
    schwab_df = schwab_df[columns_to_keep].copy()

    # Filter using your Master Key
    schwab_df = schwab_df[schwab_df['PATNO'].isin(master_df['PATNO'])]

    # 4. Filter out EVENT_IDs that are not relevant
    # REMOVE 'SC', 'ST', 'U01'
    events_to_remove = ['SC', 'ST', 'U01']
    
    # Keep rows where EVENT_ID is NOT in the remove list
    schwab_df = schwab_df[~schwab_df['EVENT_ID'].isin(events_to_remove)]

    # ---------------------------------------------------------
    # CLEANUP
    # ---------------------------------------------------------
    # Remove rows where MSEADLG is Missing (Null)
    schwab_df = schwab_df.dropna(subset=['MSEADLG'])
    print('Check for missing value:\n', schwab_df.isnull().sum())

    # Reset index for a clean dataframe
    master_df.reset_index(drop=True, inplace=True)


    # Convert INFODT to datetime
    schwab_df['INFODT'] = pd.to_datetime(schwab_df['INFODT'],format='%m/%Y')

    # Merge with Master Key to get the Anchor Date (ENROLL_DATE)
    schwab_df = pd.merge(schwab_df, master_df[['PATNO', 'ENROLL_DATE']], on='PATNO', how='left')

    return schwab_df

# --- EXECUTION ---
file_path = '../../ppmi_pd/Motor___MDS-UPDRS/Modified_Schwab___England_Activities_of_Daily_Living_14Dec2025.csv'
schwab_df = create_schwab_df(file_path)

# Calculate Timeline
schwab_df['Months_Since_Enrollment'] = (
    (schwab_df['INFODT'] - schwab_df['ENROLL_DATE']) / pd.Timedelta(days=30.44)
).round().astype('Int64')

# Rename for consistency
schwab_df = schwab_df.rename(columns={'INFODT': 'VISIT_DATE', 'MSEADLG': 'Schwab_England_Score'})

schwab_df.head(20)

Check for missing value:
 REC_ID         0
PATNO          0
EVENT_ID       0
PAG_NAME       0
INFODT         0
MSEADLG        0
LAST_UPDATE    0
dtype: int64


,REC_ID,PATNO,EVENT_ID,PAG_NAME,VISIT_DATE,Schwab_England_Score,LAST_UPDATE,ENROLL_DATE,Months_Since_Enrollment
0,IA123379,3000,V17,MODSEADL,2021-05-01,95.0,2021-06-01 00:00:00,2011-02-01,123
1,278744301,3001,BL,MODSEADL,2011-03-01,95.0,2020-06-25 16:04:30,2011-03-01,0
2,IA123380,3001,R17,MODSEADL,2021-11-01,65.0,2021-12-02 00:00:00,2011-03-01,128
3,IA123383,3001,R18,MODSEADL,2022-09-01,70.0,2022-09-30 00:00:00,2011-03-01,138
4,IA318975,3001,R19,MODSEADL,2023-08-01,70.0,2023-08-23 00:00:00,2011-03-01,149
5,294965201,3001,V01,MODSEADL,2011-05-01,95.0,2020-06-25 16:04:33,2011-03-01,2
6,310817701,3001,V02,MODSEADL,2011-08-01,95.0,2020-06-25 16:04:33,2011-03-01,5
7,322771101,3001,V03,MODSEADL,2011-11-01,90.0,2020-06-25 16:04:33,2011-03-01,8
8,341438501,3001,V04,MODSEADL,2012-03-01,95.0,2020-06-25 16:04:33,2011-03-01,12
9,362684001,3001,V05,MODSEADL,2012-09-01,95.0,2020-06-25 16:04:33,2011-03-01,18


### Neuro_QoL__Lower

In [9]:
neuro_qol_lower_df = pd.read_csv('../../ppmi_pd/Motor___MDS-UPDRS/Neuro_QoL__Lower_Extremity_Function__Mobility__-_Short_Form_14Dec2025.csv')
neuro_qol_lower_df.head(20)

,REC_ID,PATNO,EVENT_ID,PAG_NAME,INFODT,NQMOB37,NQMOB30,NQMOB26,NQMOB32,NQMOB25,NQMOB33,NQMOB31,NQMOB28,ORIG_ENTRY,LAST_UPDATE
0,736294201,3000,V15,NQOLLEFS,03/2019,5,5,5,5.0,5.0,5,5.0,5.0,04/2019,2020-06-25 16:02:24
1,IA132707,3000,V17,NQOLLEFS,05/2021,5,5,5,5.0,5.0,5,5.0,5.0,06/2021,2021-06-01 00:00:00
2,733239901,3001,V15,NQOLLEFS,03/2019,5,5,5,5.0,5.0,5,5.0,5.0,03/2019,2020-06-25 16:04:36
3,IA132708,3001,V17,NQOLLEFS,09/2021,4,5,3,4.0,4.0,3,4.0,4.0,01/2022,2022-01-06 00:00:00
4,IA132709,3001,V18,NQOLLEFS,07/2022,4,5,4,4.0,4.0,3,4.0,5.0,07/2022,2022-07-08 00:00:00
5,IA249073,3001,V19,NQOLLEFS,03/2023,4,5,4,4.0,3.0,2,4.0,5.0,03/2023,2023-03-29 00:00:00
6,IA583158,3001,V20,NQOLLEFS,09/2024,3,4,3,3.0,3.0,2,3.0,3.0,09/2024,2024-09-25 00:00:00
7,733825401,3002,V15,NQOLLEFS,03/2019,4,4,3,2.0,2.0,3,2.0,2.0,03/2019,2020-06-30 09:26:04
8,IA132710,3002,V17,NQOLLEFS,09/2021,3,2,2,2.0,2.0,1,1.0,3.0,01/2022,2022-01-06 00:00:00
9,IA132711,3002,V18,NQOLLEFS,03/2022,4,2,3,1.0,1.0,1,1.0,1.0,03/2022,2022-03-11 00:00:00


In [10]:
def create_schwab_df(file_path):
    # 1. Load the raw Schwab & England file
    schwab_df = pd.read_csv(file_path)

    # 2. Define the exact columns needed
    columns_to_keep = [
        'REC_ID',
        'PATNO',
        'EVENT_ID',
        'PAG_NAME',
        'INFODT',
        'MSEADLG',
        'LAST_UPDATE'
    ]
    
    # 3. Create the subset DataFrame
    schwab_df = schwab_df[columns_to_keep].copy()

    # Filter using your Master Key
    schwab_df = schwab_df[schwab_df['PATNO'].isin(master_df['PATNO'])]

    # 4. Filter out EVENT_IDs that are not relevant
    # REMOVE 'SC', 'ST', 'U01'
    events_to_remove = ['SC', 'ST', 'U01']
    
    # Keep rows where EVENT_ID is NOT in the remove list
    schwab_df = schwab_df[~schwab_df['EVENT_ID'].isin(events_to_remove)]

    # ---------------------------------------------------------
    # CLEANUP
    # ---------------------------------------------------------
    # Remove rows where MSEADLG is Missing (Null)
    schwab_df = schwab_df.dropna(subset=['MSEADLG'])
    print('Check for missing value:\n', schwab_df.isnull().sum())

    # Reset index for a clean dataframe
    master_df.reset_index(drop=True, inplace=True)


    # Convert INFODT to datetime
    schwab_df['INFODT'] = pd.to_datetime(schwab_df['INFODT'],format='%m/%Y')

    # Merge with Master Key to get the Anchor Date (ENROLL_DATE)
    schwab_df = pd.merge(schwab_df, master_df[['PATNO', 'ENROLL_DATE']], on='PATNO', how='left')
def create_schwab_df(file_path):
    # 1. Load the raw Schwab & England file
    schwab_df = pd.read_csv(file_path)

    # 2. Define the exact columns needed
    columns_to_keep = [
        'REC_ID',
        'PATNO',
        'EVENT_ID',
        'PAG_NAME',
        'INFODT',
        'MSEADLG',
        'LAST_UPDATE'
    ]
    
    # 3. Create the subset DataFrame
    schwab_df = schwab_df[columns_to_keep].copy()

    # Filter using your Master Key
    schwab_df = schwab_df[schwab_df['PATNO'].isin(master_df['PATNO'])]

    # 4. Filter out EVENT_IDs that are not relevant
    # REMOVE 'SC', 'ST', 'U01'
    events_to_remove = ['SC', 'ST', 'U01']
    
    # Keep rows where EVENT_ID is NOT in the remove list
    schwab_df = schwab_df[~schwab_df['EVENT_ID'].isin(events_to_remove)]

    # ---------------------------------------------------------
    # CLEANUP
    # ---------------------------------------------------------
    # Remove rows where MSEADLG is Missing (Null)
    schwab_df = schwab_df.dropna(subset=['MSEADLG'])
    print('Check for missing value:\n', schwab_df.isnull().sum())

    # Reset index for a clean dataframe
    master_df.reset_index(drop=True, inplace=True)


    # Convert INFODT to datetime
    schwab_df['INFODT'] = pd.to_datetime(schwab_df['INFODT'],format='%m/%Y')

    # Merge with Master Key to get the Anchor Date (ENROLL_DATE)
    schwab_df = pd.merge(schwab_df, master_df[['PATNO', 'ENROLL_DATE']], on='PATNO', how='left')

    return schwab_df

# --- EXECUTION ---
file_path = '../../ppmi_pd/Motor___MDS-UPDRS/Modified_Schwab___England_Activities_of_Daily_Living_14Dec2025.csv'
schwab_df = create_schwab_df(file_path)

# Calculate Timeline
schwab_df['Months_Since_Enrollment'] = (
    (schwab_df['INFODT'] - schwab_df['ENROLL_DATE']) / pd.Timedelta(days=30.44)
).round().astype('Int64')

# Rename for consistency
schwab_df = schwab_df.rename(columns={'INFODT': 'VISIT_DATE', 'MSEADLG': 'Schwab_England_Score'})

schwab_df.head(20)

Check for missing value:
 REC_ID         0
PATNO          0
EVENT_ID       0
PAG_NAME       0
INFODT         0
MSEADLG        0
LAST_UPDATE    0
dtype: int64


,REC_ID,PATNO,EVENT_ID,PAG_NAME,VISIT_DATE,Schwab_England_Score,LAST_UPDATE,ENROLL_DATE,Months_Since_Enrollment
0,IA123379,3000,V17,MODSEADL,2021-05-01,95.0,2021-06-01 00:00:00,2011-02-01,123
1,278744301,3001,BL,MODSEADL,2011-03-01,95.0,2020-06-25 16:04:30,2011-03-01,0
2,IA123380,3001,R17,MODSEADL,2021-11-01,65.0,2021-12-02 00:00:00,2011-03-01,128
3,IA123383,3001,R18,MODSEADL,2022-09-01,70.0,2022-09-30 00:00:00,2011-03-01,138
4,IA318975,3001,R19,MODSEADL,2023-08-01,70.0,2023-08-23 00:00:00,2011-03-01,149
5,294965201,3001,V01,MODSEADL,2011-05-01,95.0,2020-06-25 16:04:33,2011-03-01,2
6,310817701,3001,V02,MODSEADL,2011-08-01,95.0,2020-06-25 16:04:33,2011-03-01,5
7,322771101,3001,V03,MODSEADL,2011-11-01,90.0,2020-06-25 16:04:33,2011-03-01,8
8,341438501,3001,V04,MODSEADL,2012-03-01,95.0,2020-06-25 16:04:33,2011-03-01,12
9,362684001,3001,V05,MODSEADL,2012-09-01,95.0,2020-06-25 16:04:33,2011-03-01,18
